# ReMDM Discrete Diffusion Planner for MiniHack -- Demo Notebook

> **COMP0258 coursework submission.** This notebook is fully self-contained:
> upload only the `.ipynb` file to Google Colab and run all cells from top to
> bottom. Everything (source code, pre-trained checkpoint, ablation assets)
> is downloaded from a single public HuggingFace repo by the install cell.

The notebook does three things:

1. **Loads the pre-trained ReMDM dual-stream planner** (no training).
2. **Runs live inference** on 4 in-distribution and 3 out-of-distribution
   MiniHack environments. The marker can change environments, episode counts,
   seeds, or supply a custom `.des` scenario in Cell 0.
3. **Reproduces the RL fine-tuning ablation findings** with pre-computed
   figures and tables, plus a live visualisation of the ReMDM denoising
   process and the agent's observations.


In [ ]:
#====================================================================
# CELL 0  --  MARKER: change these to test on unseen inputs
#====================================================================

# === HuggingFace repo ===
HF_REPO_ID = "MathisW78/remdm-minihack-demo"

# === Quick mode ===
# True  -> 5 episodes per env (fast on CPU/GPU, ~2-5 min total)
# False -> 20 episodes per env (more reliable numbers, ~10-20 min)
QUICK_MODE = True
EPISODES_PER_ENV = 5 if QUICK_MODE else 20

# === Environments to evaluate (free to add or remove) ===
ID_ENVS = [
    "MiniHack-Room-Random-5x5-v0",
    "MiniHack-Room-Random-15x15-v0",
    "MiniHack-Corridor-R2-v0",
    "MiniHack-MazeWalk-9x9-v0",
]
OOD_ENVS = [
    "MiniHack-Room-Dark-15x15-v0",
    "MiniHack-Corridor-R5-v0",
    "MiniHack-MazeWalk-45x19-v0",
]

# === Optional: custom .des scenario file (set to None to skip) ===
# Provide either a path inside the downloaded snapshot (e.g. "environments/x.des")
# or an absolute Colab path you uploaded yourself.
CUSTOM_DES_FILE = None

# === Reproducibility ===
SEED = 42

# === Notes for the marker ===
# - Each MiniHack env is procedurally generated. Different SEED values
#   produce different layouts -- the model is automatically tested on
#   unseen maps every time you re-run the cells.
# - QUICK_MODE = True is enough to reproduce the qualitative finding
#   (DAgger >> baselines, OOD non-trivial). Set False for paper-grade
#   numbers.


In [ ]:
#====================================================================
# CELL 1  --  Install dependencies and download project from HuggingFace
#====================================================================

import sys, subprocess, os

# 1. System packages required to compile NLE (NetHack Learning Environment).
#    Colab usually has these, but install is idempotent.
print("[1/4] Installing system build deps for NLE...")
subprocess.run(
    "apt-get -qq update && "
    "apt-get install -y -qq cmake build-essential bison flex libbz2-dev",
    shell=True, check=False, capture_output=True,
)

# 2. Python dependencies. Pin versions loosely so Colab's pre-installed
#    torch/numpy are reused (avoids long re-installs and CUDA mismatches).
print("[2/4] Installing Python deps...")
PIP_PACKAGES = [
    "huggingface_hub>=0.24",
    "nle>=1.2.0",
    "minihack>=1.0.2",
    "pyyaml>=6.0",
    "matplotlib>=3.7",
    "pandas>=2.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + PIP_PACKAGES,
    check=True,
)

# 3. Verify NLE installed correctly. This is the highest-risk failure
#    point -- NLE compiles NetHack from C source.
print("[3/4] Verifying NLE / MiniHack import...")
try:
    import nle
    import minihack
    print(f"      OK -- NLE {nle.__version__}, MiniHack {minihack.__version__}")
except ImportError as e:
    raise RuntimeError(
        "\n\nNLE/MiniHack installation failed: " + str(e) + "\n\n"
        "Try: Runtime -> Restart runtime, then re-run this cell.\n"
        "If that fails, check the apt-get output above for missing packages.\n"
    ) from e

# 4. Download project (source code, stripped checkpoint, ablation assets)
#    from the public HuggingFace repo.
print(f"[4/4] Downloading project from {HF_REPO_ID}...")
if HF_REPO_ID == "TODO_HF_REPO_ID":
    raise RuntimeError(
        "Replace HF_REPO_ID in Cell 0 with the actual HuggingFace repo ID "
        "before running this cell."
    )

from huggingface_hub import snapshot_download
PROJECT_DIR = snapshot_download(
    repo_id=HF_REPO_ID,
    local_dir="remdm-minihack",
    repo_type="model",
)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"      OK -- project at {PROJECT_DIR}")
print()
print("Setup complete. Continue to Cell 2.")


## Project overview

We tackle **action-sequence planning in MiniHack** with a discrete diffusion
model. A 5.2M-parameter dual-stream transformer (`LocalDiffusionPlannerWithGlobal`)
generates 64-step action plans by iteratively denoising a fully masked token
sequence. It is conditioned on a 9x9 local glyph crop *and* the full 21x79
dungeon map, with a sigmoid-gated global stream and an auxiliary staircase
prediction head.

**Training:** DAgger with a BFS oracle. The model rolls out, the oracle runs
on the same seed, and an efficiency filter decides whether the oracle trajectory
is added to the replay buffer. Cross-entropy is computed only on masked
positions (MDLM ELBO).

**Inference (ReMDM):** the all-MASK plan is denoised over 10 steps with
MaskGIT-style progressive unmasking and stochastic remasking of low-confidence
committed positions. Full details in `src/diffusion/sampling.py`.

**Research question:** can RL fine-tuning improve on DAgger? We test 25
ablations across regularisation, training-signal, partial-parameter, and
data-quality variants -- the answer turns out to be **no**, and the cells
below show why.


In [ ]:
#====================================================================
# CELL 3  --  Load the pre-trained model
#====================================================================

import torch, random
import numpy as np

from src.config import load_config
from src.models.denoiser import make_model, ModelEMA

# Load default training config (matches the checkpoint)
cfg = load_config("configs/defaults.yaml", {})
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
cfg.seed = SEED

# Seed everything
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Build the model and load EMA weights
model = make_model(cfg).to(cfg.device)
ckpt = torch.load(
    "checkpoint_inference.pth",
    map_location=cfg.device,
    weights_only=False,
)
ema = ModelEMA(model, decay=cfg.ema_decay)
ema.load_state_dict(ckpt["ema_state_dict"])
ema.apply_to(model)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Device:        {cfg.device}")
print(f"Model:         LocalDiffusionPlannerWithGlobal")
print(f"Parameters:    {n_params:,}  (~{n_params/1e6:.2f}M)")
print(f"Architecture:  dual-stream CNN  +  4-layer transformer "
      f"({cfg.n_embd}d, {cfg.n_head} heads)")
print(f"Action vocab:  {cfg.action_dim} actions + MASK + PAD = "
      f"{cfg.action_dim + 2} tokens")
print(f"Plan horizon:  {cfg.seq_len} action tokens")
print(f"Denoising:     {cfg.diffusion_steps_eval} steps, "
      f"strategy={cfg.remask_strategy!r}, eta={cfg.eta}")
print(f"\nLoaded EMA weights -> ready for inference.")


## Live inference (Cells 4-6)

The next three cells **run the model end-to-end on freshly seeded MiniHack
episodes**. They are the answer to the submission guideline "give us an easy
way to test your system on unseen inputs". Each MiniHack env is procedurally
generated -- changing `SEED` in Cell 0 produces different maps, so the marker
is automatically testing on layouts the model never saw.


In [ ]:
#====================================================================
# CELL 4  --  Live inference on ID + OOD environments
#====================================================================

import time
from src.planners.inference import Evaluator, format_eval_results

# Resolve optional .des file
des_files = None
if CUSTOM_DES_FILE is not None:
    des_path = (
        CUSTOM_DES_FILE
        if os.path.isabs(CUSTOM_DES_FILE)
        else os.path.join(PROJECT_DIR, CUSTOM_DES_FILE)
    )
    if not os.path.exists(des_path):
        raise FileNotFoundError(f"CUSTOM_DES_FILE not found: {des_path}")
    des_files = [des_path]
    print(f"Custom .des file: {des_path}\n")

evaluator = Evaluator()

# In-distribution
print("=" * 60)
print(f"  In-distribution evaluation ({EPISODES_PER_ENV} eps / env)")
print("=" * 60)
t0 = time.perf_counter()
id_results = evaluator.evaluate(
    ID_ENVS, model, EPISODES_PER_ENV, cfg, cfg.device,
)
print(format_eval_results(id_results, label="In-distribution"))
print(f"Elapsed: {time.perf_counter()-t0:.1f}s")

# Out-of-distribution (zero-shot transfer)
print("\n" + "=" * 60)
print(f"  Out-of-distribution evaluation (zero-shot)")
print("=" * 60)
t0 = time.perf_counter()
ood_results = evaluator.evaluate(
    OOD_ENVS, model, EPISODES_PER_ENV, cfg, cfg.device,
    des_files=des_files,
)
print(format_eval_results(ood_results, label="Out-of-distribution"))
print(f"Elapsed: {time.perf_counter()-t0:.1f}s")

# Aggregate comparison vs paper
import pandas as pd
id_mean = sum(r["win_rate"] for r in id_results.values()) / len(id_results)
ood_mean = sum(r["win_rate"] for r in ood_results.values()) / len(ood_results)
summary = pd.DataFrame([
    {"Split": "ID  (4 envs)", "Measured %": f"{id_mean*100:5.1f}",
     "Reported (paper) %": "67.5"},
    {"Split": "OOD (3 envs)", "Measured %": f"{ood_mean*100:5.1f}",
     "Reported (paper) %": "10.0"},
])
print("\n" + "=" * 60)
print("  Reproducibility check  (live vs reported)")
print("=" * 60)
print(summary.to_string(index=False))
print()
print("Note: with QUICK_MODE=True (5 eps/env) win rates have ~20% noise.")
print("Set QUICK_MODE=False for tighter bounds.")


In [ ]:
#====================================================================
# CELL 5  --  Visualise the agent's observations during a rollout
#====================================================================

import matplotlib.pyplot as plt
from src.planners.collect import run_model_episode

DEMO_ENV = "MiniHack-Room-Random-15x15-v0"
print(f"Rolling out one stochastic episode in {DEMO_ENV}...")
result = run_model_episode(
    model, DEMO_ENV, cfg, cfg.device,
    seed=SEED, max_steps=200, stochastic=True,
)
T = result["steps"]
print(f"  steps={T}, won={result['won']}, "
      f"total_reward={result['total_reward']:.2f}")

# Pick 3 timesteps: start, middle, last
ts = [0, T // 2, max(0, T - 1)] if T >= 3 else list(range(T))
fig, axes = plt.subplots(2, len(ts), figsize=(4 * len(ts), 6))
if len(ts) == 1:
    axes = axes.reshape(2, 1)
for col, t in enumerate(ts):
    axes[0, col].imshow(result["local"][t], cmap="viridis")
    axes[0, col].set_title(f"t={t}: 9x9 local crop")
    axes[0, col].axis("off")
    axes[1, col].imshow(result["global"][t], cmap="viridis", aspect="auto")
    axes[1, col].set_title(f"t={t}: 21x79 global map")
    axes[1, col].axis("off")
fig.suptitle(
    f"{DEMO_ENV}  |  steps={T}  |  won={result['won']}",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print()
print("The local crop is the agent-centred 9x9 glyph window that the local")
print("CNN ingests. The global map is the full 21x79 dungeon that the gated")
print("global CNN sees -- the goal head is trained on the staircase position")
print("from this stream.")


In [ ]:
#====================================================================
# CELL 6  --  Visualise the ReMDM denoising process
#====================================================================
#
# Show how a fully-masked [MASK]*64 plan gets iteratively unmasked over
# K denoising steps. The model commits high-confidence positions
# (MaskGIT) and stochastically re-masks low-confidence ones (ReMDM).

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from src.diffusion.sampling import remdm_sample
from src.envs.minihack_env import make_env

# Build a real observation for conditioning
env = make_env("MiniHack-MazeWalk-9x9-v0", None, cfg)
(local, glb), _ = env.reset(seed=SEED)
env.close()

local_t = torch.from_numpy(local[None]).long().to(cfg.device)
glb_t = torch.from_numpy(glb[None]).long().to(cfg.device)

# return_analytics gives us the per-step plan snapshots
seq, path_per_step, conf_track, masked_count = remdm_sample(
    model, local_t, glb_t, cfg, cfg.device,
    physics_aware=False, return_analytics=True,
)
K = len(path_per_step)
H = cfg.seq_len
MASK = cfg.mask_token

# Build [K, H] grid: 0 = MASK, 1..12 = committed action token + 1
grid = np.zeros((K, H), dtype=np.int8)
for k, step in enumerate(path_per_step):
    grid[k] = np.where(step == MASK, 0, step + 1)

# Custom colormap: black for MASK, tab20 for actions
colors = [(0, 0, 0)] + [plt.get_cmap("tab20")(i / 12) for i in range(12)]
cmap = ListedColormap(colors)

fig, (ax_grid, ax_curve) = plt.subplots(
    2, 1, figsize=(14, 6),
    gridspec_kw={"height_ratios": [3, 1]},
    sharex=True,
)
ax_grid.imshow(grid, cmap=cmap, aspect="auto", interpolation="nearest")
ax_grid.set_ylabel("Denoising step (1..K)")
ax_grid.set_yticks(range(K))
ax_grid.set_yticklabels(range(1, K + 1))
ax_grid.set_title(
    "ReMDM denoising trajectory  --  "
    "rows = step, columns = plan position. Black = MASK"
)

ax_curve.plot(range(1, K + 1), masked_count, "o-", color="black")
ax_curve.set_ylabel("# MASK tokens")
ax_curve.set_xlabel("Plan position (0..63)")
ax_curve.set_title("MASK token count per step")
ax_curve.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final action plan ({H} tokens):")
print(seq[0].cpu().numpy().tolist())
print()
print(f"Average per-step confidence on unmasked tokens: "
      f"{np.mean(conf_track):.3f}")


## Imitation learning context

The next cell shows the headline imitation-learning result from the paper
(Table 1). The DAgger ReMDM model -- the same architecture you just ran
above -- achieves **67.5% in-distribution win rate** versus **<6% for all
model-free baselines** (PPO, A2C, DQN, PPO-RNN) and **3% for a CNN+MLP
behaviour cloning baseline**. Zero-shot OOD transfer is **10%**, again an
order of magnitude better than the baselines.


In [ ]:
#====================================================================
# CELL 7  --  Imitation learning baselines (Table 1, hardcoded)
#====================================================================

import pandas as pd

table1 = pd.DataFrame([
    ("PPO",                  0.5,  400,  0.0,  767),
    ("A2C",                  6.0,  384,  0.5,  760),
    ("DQN",                  4.5,  388,  1.3,  762),
    ("PPO-RNN",              4.0,  386,  0.5,  765),
    ("CNN+MLP (Offline BC)", 3.0,  395,  0.5,  765),
    ("ReMDM (Offline BC)",   58.2, None, 7.0,  None),
    ("ReMDM (DAgger)",       67.5, 132,  10.0, 402),
], columns=["Method", "ID Win%", "ID Steps", "OOD Win%", "OOD Steps"])
print("Table 1 -- MiniHack imitation learning baselines")
print(table1.to_string(index=False))


## RL fine-tuning ablation findings

The pre-computed figures and tables below are the main research contribution.
We tried **25 RL fine-tuning ablations** starting from the DAgger checkpoint
shown above and found that **none of them meaningfully improved** on the
67.5% pretrained baseline. The figures explain *why*.


In [ ]:
#====================================================================
# CELL 8  --  Pre-computed ablation figures
#====================================================================

from IPython.display import Image, display, Markdown

FIGURES = [
    ("assets/score_comparison.png",
     "**Score comparison.** No ablation matches the pretrained DAgger "
     "checkpoint (0.675). The best ablation reaches ~0.69 -- statistically "
     "indistinguishable from the pretrained baseline."),
    ("assets/score_delta.png",
     "**Score delta.** Most ablations land at or below the baseline RL run. "
     "`normalized_adv` collapses catastrophically."),
    ("assets/group_comparison.png",
     "**Group comparison.** Group C (partial-parameter methods: frozen "
     "backbone, head-only, attention-only) is universally worst."),
    ("assets/per_env_delta.png",
     "**Per-environment delta.** Easy envs (Room-Random) are already at "
     "ceiling; hard envs (MazeWalk) see no improvement -- the deltas are "
     "concentrated in noise."),
    ("assets/grad_alignment.png",
     "**Gradient alignment.** Cosine similarity between RL and BC gradients "
     "is near zero across the network -- the RL signal is effectively "
     "uninformative compared to the imitation signal."),
    ("assets/gradient_conflict_map.png",
     "**Gradient conflict map.** Per-layer alignment heatmap. Confirms that "
     "RL and BC objectives compete rather than cooperate."),
    ("assets/repr_drift.png",
     "**Representation drift.** Internal activations diverge from the "
     "pretrained baseline as RL fine-tuning progresses."),
    ("assets/diagnosis_decision_tree.png",
     "**Diagnosis decision tree.** Hypothesis attribution from the ablation "
     "results -- which failure mode each verdict supports."),
]

for path, caption in FIGURES:
    if os.path.exists(path):
        display(Markdown(caption))
        display(Image(filename=path))
    else:
        print(f"(missing: {path})")


In [ ]:
#====================================================================
# CELL 9  --  Pre-computed ablation tables
#====================================================================

import pandas as pd

main_results = pd.read_csv("assets/main_results.csv")
print("=" * 70)
print("  main_results.csv  --  all 25 ablations, sorted by score")
print("=" * 70)
print(main_results.sort_values("Score", ascending=False).to_string(index=False))

DAGGER_PRETRAINED = 0.675
best_method = main_results.loc[main_results["Score"].idxmax(), "Method"]
best_score = main_results["Score"].max()
print()
print(f"Best ablation:        {best_method} = {best_score:.4f}")
print(f"DAgger pretrained:    {DAGGER_PRETRAINED:.4f}")
print(f"Best vs pretrained:   {best_score - DAGGER_PRETRAINED:+.4f}")
print()

verdicts = pd.read_csv("assets/hypothesis_verdicts.csv")
print("=" * 70)
print("  hypothesis_verdicts.csv  --  per-ablation verdict")
print("=" * 70)
print(
    verdicts[["Method", "Group", "Score", "Verdict"]]
    .to_string(index=False)
)


## Conclusions

1. **ReMDM DAgger is a strong imitation learner**: 67.5% in-distribution win
   rate and 10% zero-shot OOD transfer -- an order of magnitude above PPO,
   A2C, DQN, and CNN+MLP baselines.

2. **No RL ablation we tried meaningfully improves on the DAgger checkpoint.**
   Across 25 variants spanning regularisation (KL, EWC, LLRD, LoRA, mixed
   replay, trust region), training signal (entropy, PCGrad, advantage
   clipping, t-curriculum, BC-on-wins, low-t), partial-parameter
   (frozen backbone, head-only, attention-only, FFN-only, layer-ablation),
   and data quality (reward filtering, running stats, action diversity,
   reward model) -- the best result lands within noise of the pretrained
   baseline, and several collapse catastrophically.

3. **Double intractability.** Two issues compound:
   - Standard policy gradients are intractable for masked discrete
     diffusion (no closed-form `log pi_theta` for a sampled action sequence).
   - The Return-Weighted ELBO surrogate degenerates because episode returns
     in MiniHack lack sufficient variance to discriminate trajectories,
     yielding near-zero gradient cosine similarity with the BC objective.

4. **Group C (partial-parameter) is universally worst.** Restricting which
   weights move does not protect against the underlying objective mismatch
   -- it just removes capacity.

5. **Discrete diffusion planners look like fundamentally imitation-learning
   architectures** in their current form. RL fine-tuning, as commonly
   formulated, does not transfer.

6. **Open problem:** designing a tractable RL objective for masked discrete
   diffusion. Promising directions include Q-guided remasking, SDE
   reformulations of the masked diffusion process, and reward-weighted
   importance sampling that avoids the surrogate-gradient bottleneck.
